<a href="https://colab.research.google.com/github/Gene7Ai/genai-tps/blob/main/TP4/TP4_Autoregressive_Models_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

For each practical exercise (TP), please work in groups of two or three. Then, create a private GitHub repository and add me (my GitHub is arthur-75) to your project.

# Autoregressive Models (GPT)

## 1. What is an Autoregressive Model?

Autoregressive models generate sequences by predicting one token at a time, conditioned on all previously generated tokens. Formally, for a sequence of tokens:

$$x = (x_1, x_2, x_3, \dots, x_n)$$

an autoregressive model factorizes the probability distribution as:

$$P(x) = \prod_{t=1}^{n} P(x_t \mid x_1, x_2, \dots, x_{t-1})$$

This means each token xt depends explicitly only on previously observed tokens, which makes it ideal for sequential text generation tasks.

Formally, the conditional probability of predicting a token $x_t$ given previous tokens is expressed as:

$$P(x_t \mid x_{<t}) = \mathrm{Softmax}(W \cdot h_{t-1})$$

where:

x<t denotes all tokens generated before the current token t.

h represents hidden states produced by the Transformer layers.

W represents vocabulary vectors.

### Data Preprocessing and Tokenization

Data Cleaning: Ensure text is consistent by spacing punctuation, lowercasing, and removing unwanted characters.

Tokenization: Convert text into discrete tokens. Each unique word gets a unique integer index.

Vocabulary Creation: Build vocabulary from the most frequent words (usually limited to a fixed size, e.g., 10,000).

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.nn.functional import cross_entropy
import numpy as np
import re, string
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter
import json
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zynicide/wine-reviews" )
# Parameters
VOCAB_SIZE = 10000
MAX_LEN = 80
EMBEDDING_DIM = 256
N_HEADS = 2
FF_DIM = 256
BATCH_SIZE = 32
EPOCHS = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load the data
with open(path+"/winemag-data-130k-v2.json") as json_data:
    wine_data = json.load(json_data)
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]
# Tokenizer
class SimpleTokenizer:
    def __init__(self, texts, vocab_size):
        self.vocab_size = vocab_size
        self.counter = Counter()
        for text in texts:
            self.counter.update(text.split())
        self.vocab = [word for word, _ in self.counter.most_common(vocab_size - 2)]
        self.word2idx = {word: idx + 2 for idx, word in enumerate(self.vocab)}
        self.word2idx["<pad>"] = 0
        self.word2idx["<unk>"] = 1

    def encode(self, text):
        return [self.word2idx.get(word, 1) for word in text.split()]

# Data preparation
def pad_punctuation(s):
    s = re.sub(f"([{string.punctuation}])", r" \1 ", s)
    s = re.sub(" +", " ", s)
    return s.lower()


# Create tokenizer
tokenizer = SimpleTokenizer(filtered_data, VOCAB_SIZE)

Using Colab cache for faster access to the 'wine-reviews' dataset.


Constructing the Dataset

Input-target pairs: Prepare data such that each input sequence is paired with the same sequence shifted by one token.

Input sequence: (x1,x2,...,xn−1)

Target sequence: (x2,x3,...,xn)

*(figure from the original TP, see the Google Doc)*

In [2]:
# Dataset class
class WineDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.tokenizer.encode(self.texts[idx])[:self.max_len+1]
        padding = [self.tokenizer.word2idx["<pad>"]] * (self.max_len + 1 - len(tokens))
        tokens += padding
        return torch.tensor(tokens[:-1]), torch.tensor(tokens[1:])

train_dataset = WineDataset([pad_punctuation(t) for t in filtered_data], tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

### Model Definition

An autoregressive model (like GPT) typically consists of:

Embedding Layer: Maps discrete tokens into continuous vector embeddings.

Positional Encoding: Adds positional information to token embeddings to retain sequence order.

Transformer Block:

Self-attention: Captures dependencies between tokens.
 Formally, self-attention computes:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Where:

Q, K and V represent Query, Key, and Value embeddings.

$d_k$ is the dimension of the keys.

The mechanics of an attention head

*(figure from the original TP, see the Google Doc)*

*(figure from the original TP, see the Google Doc)* — a multihead attention layer with four heads

Feedforward Network: Introduces non-linearity and transforms attention outputs:

$$\mathrm{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2$$

*(figure from the original TP, see the Google Doc)* — a transformer block

Layer normalization and residual connections are added to stabilize training and help information flow.

## Causal Attention Masking

To maintain autoregressive behavior (no peeking into the future tokens), you need a causal mask, ensuring that each position in the sequence can only attend to itself and all previous positions, not future positions.

$$M_{i,j} = \begin{cases} 0 & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

This masking ensures the model predicts the next token based only on previously seen tokens.

In [3]:
class GPTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        seq_len = x.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len)).to(x.device)
        mask = mask.masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, 0.0)

        attn_output, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + attn_output)
        ffn_output = self.ffn(x)
        return self.ln2(x + ffn_output)

GPTModel

Finally a dense layer to represent vocabulary logits

*(figure from the original TP, see the Google Doc)* — the full GPT model

In [4]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_len, num_heads, ff_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(max_len, embed_dim)
        self.transformer = GPTBlock(embed_dim, num_heads, ff_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding(x) + self.pos_embedding(positions)
        x = self.transformer(x)
        return self.fc(x)

### Training Objective

Autoregressive models use cross-entropy loss, defined as:

$$\mathrm{Loss}(x, y) = -\sum_i y_i \cdot \log(\hat{x}_i)$$

where:

y is the true next token distribution.

x is the model’s predicted probabilities.

### Training Loop

Forward pass: Provide input sequence, compute logits.

Calculate loss between predicted tokens and actual next tokens.

Perform backpropagation to update weights.

In [5]:
# Training loop
# Training
model = GPTModel(VOCAB_SIZE, EMBEDDING_DIM, MAX_LEN, N_HEADS, FF_DIM).to(DEVICE)
optimizer = Adam(model.parameters(), lr=0.0001)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = cross_entropy(logits.view(-1, VOCAB_SIZE), y_batch.view(-1), ignore_index=tokenizer.word2idx['<pad>'])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}")

Epoch 1/1: 100%|██████████| 4060/4060 [1:28:11<00:00,  1.30s/it]

Epoch 1, Training Loss: 4.0084


### Load the pretrained model

The teacher's model is in the drive (`cours 3 AR/models`). `gpt-cpu.pt` is the same as `gpt.pt` but saved on CPU.

In [6]:
import os
import gdown

os.makedirs("models", exist_ok=True)
if not os.path.exists("models/gpt-cpu.pt"):
    gdown.download(id="1YNUlvuxmC8_pDeA71OUnn8mlqk91QWA_", output="models/gpt-cpu.pt")
model.load_state_dict(torch.load("models/gpt-cpu.pt", map_location=DEVICE))
model.eval()

Downloading...
From: https://drive.google.com/uc?id=1YNUlvuxmC8_pDeA71OUnn8mlqk91QWA_
To: /content/models/gpt-cpu.pt
100%|██████████| 22.2M/22.2M [00:00<00:00, 57.5MB/s]


GPTModel(
  (embedding): Embedding(10000, 256)
  (pos_embedding): Embedding(80, 256)
  (transformer): GPTBlock(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (ffn): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
    )
    (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
    (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (fc): Linear(in_features=256, out_features=10000, bias=True)
)

# Sampling and Text Generation

### Autoregressive Sampling :

Once the model is trained, generating text is done token-by-token, starting from a provided prompt.

Prompt Initialization:

Convert the initial textual prompt into token IDs.

Prediction Loop: For each step t from the current length to the desired length:

Generate probabilities for the next token:

$$P(x_t \mid x_{<t}) = \mathrm{softmax}\left(\frac{\mathrm{logits}(x_{<t})}{T}\right)$$

Here, T is the temperature controlling randomness:

Lower T (e.g., 0.5) makes the output deterministic.

Higher T (e.g., 1.5) makes it more creative/random.

Sample Token:

From the probability distribution generated by the softmax, sample the next token:

$$x_t \sim P(x_t \mid x_{<t})$$

Repeat Prediction:

Append this new token to your input and repeat until reaching the desired length or the end token.

In [7]:
def generate_text(model, tokenizer, start_prompt, max_tokens=80, temperature=1.0):
    model.eval()
    tokens = tokenizer.encode(start_prompt)
    tokens = tokens[:MAX_LEN]
    generated = tokens.copy()

    with torch.no_grad():
        for _ in range(max_tokens):
            input_tensor = torch.tensor([generated[-MAX_LEN:]], device=DEVICE)
            logits = model(input_tensor)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

            if next_token == tokenizer.word2idx['<pad>']:
                break

            generated.append(next_token)

    idx2word = {idx: word for word, idx in tokenizer.word2idx.items()}
    generated_text = ' '.join(idx2word.get(idx, '<unk>') for idx in generated)
    return generated_text

# Example usage:
prompt = "wine review : US"
generated_text = generate_text(model, tokenizer, prompt, max_tokens=50, temperature=0.8).strip()
print("Generated text:\n", generated_text)

Generated text:
 wine review : US <unk> <unk> <unk> <unk> <unk> : <unk> style red blend : a bouquet of cherry , plum , black fruit flavors sporting its crushed flavors also show some burn along with herbaceous , leafy notes . <unk> <unk> style <unk> style blend : this displays weight and the wine <unk>
